# Exercise 2: Electric power consumption

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
POWER_SOURCE = DATA_DIR / "household_power_consumption.txt"
if not POWER_SOURCE.exists():
    POWER_SOURCE = "https://assets.01-edu.org/ai-branch/piscine-ai/household_power_consumption.txt"

df = pd.read_csv(POWER_SOURCE, sep=";", low_memory=False)
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [2]:
df.drop(["Time", "Sub_metering_2", "Sub_metering_3"], axis=1, inplace=True)
df = df.assign(Date=pd.to_datetime(df["Date"], format="%d/%m/%Y").to_numpy(dtype="datetime64[ns]"))
df.set_index("Date", inplace=True)

def update_types(frame):
    return frame.apply(pd.to_numeric, errors="coerce")

df = update_types(df)
df.head().index

DatetimeIndex(['2006-12-16', '2006-12-16', '2006-12-16', '2006-12-16',
               '2006-12-16'],
              dtype='datetime64[ns]', name='Date', freq=None)

In [3]:
df.dtypes

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
dtype: object

In [4]:
overview = df.describe()
overview

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1
count,2.049280e+06,2.049280e+06,2.049280e+06,2.049280e+06,2.049280e+06
mean,1.091615e+00,1.237145e-01,2.408399e+02,4.627759e+00,1.121923e+00
std,1.057294e+00,1.127220e-01,3.239987e+00,4.444396e+00,6.153031e+00
min,7.600000e-02,0.000000e+00,2.232000e+02,2.000000e-01,0.000000e+00
25%,3.080000e-01,4.800000e-02,2.389900e+02,1.400000e+00,0.000000e+00
50%,6.020000e-01,1.000000e-01,2.410100e+02,2.600000e+00,0.000000e+00
75%,1.528000e+00,1.940000e-01,2.428900e+02,6.400000e+00,0.000000e+00
max,1.112200e+01,1.390000e+00,2.541500e+02,4.840000e+01,8.800000e+01


In [5]:
missing_before = int(df.isna().sum().sum())
df.dropna(inplace=True)
missing_after = int(df.isna().sum().sum())
df.loc[:, "Sub_metering_1"] = (df["Sub_metering_1"] + 1) * 0.06
pd.Series({"missing_before": missing_before, "missing_after": missing_after})

missing_before    129895
missing_after          0
dtype: int64

In [6]:
filtered_df = df[(df.index >= "2008-12-27") & (df["Voltage"] >= 242)]
print(len(filtered_df))
print(filtered_df.iloc[:, :2].head().to_markdown())

449667
| Date                |   Global_active_power |   Global_reactive_power |
|:--------------------|----------------------:|------------------------:|
| 2008-12-27 00:00:00 |                 0.996 |                   0.066 |
| 2008-12-27 00:00:00 |                 1.076 |                   0.162 |
| 2008-12-27 00:00:00 |                 1.064 |                   0.172 |
| 2008-12-27 00:00:00 |                 1.07  |                   0.174 |
| 2008-12-27 00:00:00 |                 0.804 |                   0.184 |


In [7]:
row_88888 = df.iloc[88887]
row_88888

Global_active_power        0.254
Global_reactive_power      0.000
Voltage                  238.100
Global_intensity           1.200
Sub_metering_1             0.060
Name: 2007-02-16 00:00:00, dtype: float64

In [8]:
max_power_date = df["Global_active_power"].idxmax()
max_power_date

Timestamp('2009-02-22 00:00:00')

In [9]:
sorted_df = df.sort_values(by=["Global_active_power", "Voltage"], ascending=[False, True]).iloc[:, :3]
print(sorted_df.tail().to_markdown())

| Date                |   Global_active_power |   Global_reactive_power |   Voltage |
|:--------------------|----------------------:|------------------------:|----------:|
| 2008-08-28 00:00:00 |                 0.076 |                       0 |    234.88 |
| 2008-08-28 00:00:00 |                 0.076 |                       0 |    235.18 |
| 2008-08-28 00:00:00 |                 0.076 |                       0 |    235.4  |
| 2008-08-28 00:00:00 |                 0.076 |                       0 |    235.64 |
| 2008-08-12 00:00:00 |                 0.076 |                       0 |    236.5  |


In [10]:
daily_average = df.groupby(level=0)["Global_active_power"].mean()
daily_average

Date
2006-12-16    3.053475
2006-12-17    2.354486
2006-12-18    1.530435
2006-12-19    1.157079
2006-12-20    1.545658
                ...   
2010-11-22    1.417733
2010-11-23    1.095511
2010-11-24    1.247394
2010-11-25    0.993864
2010-11-26    1.178230
Name: Global_active_power, Length: 1433, dtype: float64